# data process

## download the data

In [188]:
# Kaggle Notebook 数据读取模板
from pathlib import Path
import pandas as pd

def load_housing_data():
    data_path = Path("/kaggle/input/housing-data/housing_data") 
    dfs = {
        "train_price": pd.read_csv(data_path / "train_price.csv"),
        "train_rent": pd.read_csv(data_path / "train_rent.csv"),
        "test_price": pd.read_csv(data_path / "test_price.csv"),
        "test_rent": pd.read_csv(data_path / "test_rent.csv")
    }

    return dfs

housing = load_housing_data()

/tmp/ipykernel_37/951388431.py:8: DtypeWarning: Columns (3,32,34,43,46,49,51) have mixed types. Specify dtype option on import or set low_memory=False.
  "train_price": pd.read_csv(data_path / "train_price.csv"),
/tmp/ipykernel_37/951388431.py:9: DtypeWarning: Columns (23) have mixed types. Specify dtype option on import or set low_memory=False.
  "train_rent": pd.read_csv(data_path / "train_rent.csv"),
/tmp/ipykernel_37/951388431.py:10: DtypeWarning: Columns (4,32) have mixed types. Specify dtype option on import or set low_memory=False.
  "test_price": pd.read_csv(data_path / "test_price.csv"),


## data cleaning

### test_price

In [189]:
test_price = housing['test_price'].copy()
test_price.columns

Index(['ID', '城市', '区域', '板块', '环线', '房屋户型', '所在楼层', '建筑面积', '套内面积', '房屋朝向',
       '建筑结构', '装修情况', '梯户比例', '配备电梯', '别墅类型', '交易时间', '交易权属', '上次交易', '房屋用途',
       '房屋年限', '产权所属', '抵押信息', '房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', 'lon',
       'lat', '年份', '区县', '板块_comm', '环线位置', '物业类别', '建筑年代', '开发商', '房屋总数',
       '楼栋总数', '物业公司', '绿 化 率', '容 积 率', '物 业 费', '建筑结构_comm', '物业办公电话',
       '产权描述', '供水', '供暖', '供电', '燃气费', '供热费', '停车位', '停车费用', 'coord_x',
       'coord_y', '客户反馈'],
      dtype='object')

In [190]:
test_price['环线'] = test_price['环线'].fillna('未知')

In [191]:
test_price.rename(columns={
    '建筑面积': '建筑面积（㎡）',
    '绿 化 率': '绿化率（%）',
    '容 积 率': '容积率（倍）',
    '物 业 费': '物业费（元/月/㎡）',
    '燃气费': '燃气费（元/m³）',
    '供热费': '供热费（元/㎡）',
    '停车位': '停车位（个）',
    '停车费用': '停车费用（元）',
}, inplace = True)
test_price['绿化率（%）'] = test_price['绿化率（%）'].astype(str).str.replace('%', '', regex=False)
test_price['绿化率（%）'] = pd.to_numeric(test_price['绿化率（%）'], errors='coerce')
test_price['建筑面积（㎡）'] = test_price['建筑面积（㎡）'].astype(str).str.replace('㎡', '', regex=False)
test_price['建筑面积（㎡）'] = pd.to_numeric(test_price['建筑面积（㎡）'], errors='coerce')
test_price['燃气费（元/m³）'] = test_price['燃气费（元/m³）'].astype(str).str.replace('元/m³', '', regex=False)
test_price['供热费（元/㎡）'] = test_price['供热费（元/㎡）'].astype(str).str.replace('元/㎡', '', regex=False)
test_price['物业费（元/月/㎡）'] = test_price['物业费（元/月/㎡）'].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
# [^\d\.-]匹配所有不是数字、不是小数点、不是减号的字符

In [192]:
# 将区间数据用平均值代替
import pandas as pd
import numpy as np

cols = ['物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）']
for col in cols:
    test_price[col] = test_price[col].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
    test_price[col] = test_price[col].replace('', np.nan)
  
    def parse_range(x):
        if pd.isna(x):
            return np.nan
        parts = x.split('-')
        return (float(parts[0]) + float(parts[1])) / 2 if len(parts) > 1 else float(parts[0])
    
    test_price[col] = test_price[col].apply(parse_range)

test_price[cols].head()

/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.11/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）
0,2.80,2.61,NaN
1,1.80,2.61,30.0
2,1.58,2.61,30.0
3,0.50,2.61,30.0
4,1.61,2.62,27.0


In [193]:
cols_to_fix = ['绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '停车费用（元）']

for col in cols_to_fix:
    test_price[col] = pd.to_numeric(test_price[col], errors='coerce')
    
# 填补缺失值
for col in cols_to_fix:
    median_value = test_price[col].median()
    test_price[col] = test_price[col].fillna(median_value)
    print(f"{col} 缺失值已用中位数 {median_value:.2f} 填补完成")

test_price[cols_to_fix].describe()


绿化率（%） 缺失值已用中位数 35.00 填补完成
容积率（倍） 缺失值已用中位数 2.30 填补完成
物业费（元/月/㎡） 缺失值已用中位数 1.76 填补完成
燃气费（元/m³） 缺失值已用中位数 2.61 填补完成
供热费（元/㎡） 缺失值已用中位数 27.00 填补完成
停车位（个） 缺失值已用中位数 650.00 填补完成
停车费用（元） 缺失值已用中位数 250.00 填补完成


,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）,停车费用（元）
count,34017.000000,34017.000000,34017.000000,34017.000000,34017.000000,34017.000000,34017.000000
mean,36.726767,2.527234,2.246702,2.725055,24.931722,983.607255,275.290913
std,179.606810,1.330623,3.564188,0.474077,7.594384,1195.807547,177.715119
min,0.010000,0.020000,0.200000,0.400000,0.010000,1.000000,0.000000
25%,30.000000,2.000000,1.300000,2.610000,27.000000,400.000000,150.000000
50%,35.000000,2.300000,1.760000,2.610000,27.000000,650.000000,250.000000
75%,35.000000,2.780000,2.380000,3.000000,27.000000,1100.000000,300.000000
max,10500.000000,35.000000,76.450000,5.000000,50.000000,8700.000000,2300.000000


In [194]:
# 替换异常值（IQR 方法）
for col in cols_to_fix:
    Q1 = test_price[col].quantile(0.25)
    Q3 = test_price[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    median_value = test_price[col].median()
 
    test_price[col] = test_price[col].mask((test_price[col] < lower_bound) | (test_price[col] > upper_bound), median_value)

test_price[cols_to_fix].describe()

,绿化率（%）,容积率（倍）,物业费（元/月/㎡）,燃气费（元/m³）,供热费（元/㎡）,停车位（个）,停车费用（元）
count,34017.000000,34017.000000,34017.000000,34017.000000,34017.0,34017.000000,34017.000000
mean,33.982627,2.265288,1.779100,2.773272,27.0,694.619337,245.307728
std,3.128575,0.573577,0.737006,0.315682,0.0,466.903167,94.434055
min,23.000000,0.840000,0.200000,2.075000,27.0,1.000000,0.000000
25%,31.000000,2.000000,1.300000,2.610000,27.0,400.000000,150.000000
50%,35.000000,2.300000,1.760000,2.610000,27.0,650.000000,250.000000
75%,35.000000,2.500000,2.050000,3.000000,27.0,760.000000,250.000000
max,42.400000,3.910000,3.995000,3.565000,27.0,2150.000000,500.000074


### test_rent

In [195]:
test_rent = housing['test_rent'].copy()
test_rent.drop(columns=['朝向','交易时间','车位','用水','用电','采暖','租期','配套设施','年份','物业类别','建筑年代','开发商','房屋总数','楼栋总数','物业公司','建筑结构','物业办公电话','产权描述','供水','供暖','供电','coord_x','coord_y','客户反馈','停车费用'], inplace=True)

In [196]:
test_rent['环线位置'] = test_rent['环线位置'].fillna('未知')
test_rent['装修'] = test_rent['装修'].fillna('非精装修')
test_rent['付款方式'] = test_rent['付款方式'].fillna('未知')
test_rent['燃气'] = test_rent['燃气'].fillna('未知')

In [197]:
test_rent.rename(columns={
    '面积': '面积（㎡）',
    '绿 化 率': '绿化率（%）',
    '容 积 率': '容积率（倍）',
    '物 业 费': '物业费（元/月/㎡）',
    '燃气费': '燃气费（元/m³）',
    '供热费': '供热费（元/㎡）',
    '停车位': '停车位（个）'
}, inplace = True)

test_rent['绿化率（%）'] = test_rent['绿化率（%）'].astype(str).str.replace('%', '', regex=False)
test_rent['绿化率（%）'] = pd.to_numeric(test_price['绿化率（%）'], errors='coerce')
test_rent['面积（㎡）'] = test_rent['面积（㎡）'].astype(str).str.replace('㎡', '', regex=False)
test_rent['燃气费（元/m³）'] = test_rent['燃气费（元/m³）'].astype(str).str.replace('元/m³', '', regex=False)
test_rent['供热费（元/㎡）'] = test_rent['供热费（元/㎡）'].astype(str).str.replace('元/㎡', '', regex=False)
test_rent['物业费（元/月/㎡）'] = test_rent['物业费（元/月/㎡）'].astype(str).str.replace(r'[^\d\.-]', '', regex=True)

In [198]:
# 将区间数据用平均值代替
cols = ['物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）']
for col in cols:
    test_rent[col] = test_rent[col].astype(str).str.replace(r'[^\d\.-]', '', regex=True)
    test_rent[col] = test_rent[col].replace('', np.nan)
  
    def parse_range(x):
        if pd.isna(x):
            return np.nan
        parts = x.split('-')
        return (float(parts[0]) + float(parts[1])) / 2 if len(parts) > 1 else float(parts[0])
    
    test_rent[col] = test_rent[col].apply(parse_range)

In [199]:
cols_to_fix1 = ['绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）']

for col in cols_to_fix1:
    test_rent[col] = pd.to_numeric(test_rent[col], errors='coerce')
    
# 填补缺失值
for col in cols_to_fix1:
    median_value = test_rent[col].median()
    test_rent[col] = test_rent[col].fillna(median_value)
    print(f"{col} 缺失值已用中位数 {median_value:.2f} 填补完成")

绿化率（%） 缺失值已用中位数 35.00 填补完成
容积率（倍） 缺失值已用中位数 2.50 填补完成
物业费（元/月/㎡） 缺失值已用中位数 2.05 填补完成
燃气费（元/m³） 缺失值已用中位数 2.96 填补完成
供热费（元/㎡） 缺失值已用中位数 25.00 填补完成
停车位（个） 缺失值已用中位数 700.00 填补完成


In [200]:
# 替换异常值（IQR 方法）
cols_to_fix2 = ['容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）',  '停车位（个）']
for col in cols_to_fix2:
    Q1 = test_rent[col].quantile(0.25)
    Q3 = test_rent[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    median_value = test_rent[col].median()
 
    test_rent[col] = test_rent[col].mask((test_rent[col] < lower_bound) | (test_rent[col] > upper_bound), median_value)

# handling text and categorical attributes

## test_price

In [201]:
# 环线编码
ring_map = {
    '内环内': 1,
    '内环至中环': 2,
    '中环至外环': 3,
    '内环至外环': 3,  # 内环至外环和中环至外环视为同一层级
    '二环内': 1,
    '二至三环': 2,
    '三至四环': 3,
    '四至五环': 4,
    '五至六环': 5,
    '六环外': 6,
    '外环外': 6,
    '未知': 0
}

test_price['环线编码'] = test_price['环线'].map(ring_map)
test_price['环线编码'].fillna(0, inplace=True)
test_price[['环线', '环线编码']].head(10)


/tmp/ipykernel_37/2585441684.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_price['环线编码'].fillna(0, inplace=True)


,环线,环线编码
0,二至三环,2
1,五至六环,5
2,五至六环,5
3,六环外,6
4,二环内,1
5,六环外,6
6,四至五环,4
7,四至五环,4
8,未知,0
9,二至三环,2


In [202]:
# 楼层处理
test_price['所在楼层'] = test_price['所在楼层'].str.replace(r'\(.*\)', '', regex=True)
test_price['所在楼层'] = test_price['所在楼层'].str.strip()
floor_map = {
    '地下室': 0,
    '底层': 1,
    '低楼层': 2,
    '中楼层': 3,
    '高楼层': 4,
    '顶层': 5
}

test_price['楼层编码'] = test_price['所在楼层'].map(floor_map)

In [203]:
# 配备电梯处理
test_price['有电梯'] = test_price['配备电梯'].map({'无': 0, '有': 1})

test_price.drop(columns=['环线','所在楼层','配备电梯'], inplace=True)

In [204]:
import re
# 创建一个处理房屋户型的函数
def parse_layout_v2(layout):
    layout = str(layout)  # 确保传入的是字符串类型
    rooms = {'室': 0, '厅': 0, '厨': 0, '卫': 0, '房间': 0}  # 初始化各个房间数量
    match = re.findall(r'(\d+)(室|厅|厨|卫|房间)', layout)  # 匹配不同类型的房间

    for num, type_ in match:
        rooms[type_] = int(num)  # 将房间数赋给对应类型

    return rooms['房间'], rooms['室'], rooms['厅'], rooms['厨'], rooms['卫']

# 确保 '房屋户型' 列的数据为字符串格式，并填充缺失值
test_price['房屋户型'] = test_price['房屋户型'].fillna('未知').astype(str)

# 应用到数据
test_price[['房间数', '室数', '厅数', '厨数', '卫数']] = test_price['房屋户型'].apply(lambda x: pd.Series(parse_layout_v2(x)))

# 检查处理后的数据
print(test_price[['房屋户型', '房间数', '室数', '厅数', '厨数', '卫数']].head())


       房屋户型  房间数  室数  厅数  厨数  卫数
0  3室2厅1厨2卫    0   3   2   1   2
1  2室1厅1厨1卫    0   2   1   1   1
2  3室1厅1厨2卫    0   3   1   1   2
3  2室1厅1厨1卫    0   2   1   1   1
4  3室2厅1厨2卫    0   3   2   1   2


In [205]:
# 建筑结构编码
test_price['建筑结构'].fillna('未知结构', inplace=True)
structure_map = {
    '未知结构': 0,
    '钢混结构': 1,
    '钢结构': 2,
    '混合结构': 3,
    '框架结构': 4,  
    '砖混结构': 5,
    '砖木结构': 6,
}
test_price['建筑结构编码'] = test_price['建筑结构'].map(structure_map)
test_price[['建筑结构', '建筑结构编码']].head(10)

/tmp/ipykernel_37/54140488.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_price['建筑结构'].fillna('未知结构', inplace=True)


,建筑结构,建筑结构编码
0,钢混结构,1
1,钢混结构,1
2,混合结构,3
3,钢混结构,1
4,混合结构,3
5,混合结构,3
6,钢混结构,1
7,混合结构,3
8,钢混结构,1
9,钢混结构,1


## test_rent

In [206]:
# 装修
test_rent['精装修'] = test_rent['装修'].map({'非精装修': 0, '精装修': 1})

# 付款方式
print(test_rent['付款方式'].unique())
test_rent['付款方式'] = test_rent['付款方式'].str.replace(r'http\S+', '未知', regex=True)
print(test_rent['付款方式'].unique())
test_rent = pd.get_dummies(test_rent, columns=['付款方式'], drop_first=True)

['未知' '月付价' '季付价' '半年付价' '年付价' '双月付价']
['未知' '月付价' '季付价' '半年付价' '年付价' '双月付价']


In [207]:
# 租赁方式
test_rent['整租'] = test_rent['租赁方式'].map({'合租': 0, '整租': 1})
# 电梯
test_rent['有电梯'] = test_rent['电梯'].map({'无': 0, '有': 1})
# 燃气
test_rent['有燃气'] = test_rent['燃气'].map({'无': 0,'未知': 0, '有': 1})
# 环线编码
ring_map = {
    '内环内': 1,
    '内环至中环': 2,
    '中环至外环': 3,
    '内环至外环': 3,  
    '二环内': 1,
    '二至三环': 2,
    '三至四环': 3,
    '四至五环': 4,
    '五至六环': 5,
    '六环外': 6,
    '外环外': 6,
    '未知': 0
}
test_rent['环线编码'] = test_rent['环线位置'].map(ring_map)
test_rent['环线编码'].fillna(0, inplace=True)

/tmp/ipykernel_37/579995459.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_rent['环线编码'].fillna(0, inplace=True)


In [208]:
# 定义楼层区间
def convert_floor(floor_data):
    floor_data = str(floor_data).strip()
    # 如果是地下室
    if '地下' in str(floor_data):
        return '地下室'
    # 处理 'x/y层' 格式的楼层数据
    match = re.match(r'(\d+)/(\d+)层', str(floor_data))
    if match:
        floor_num = int(match.group(1))
        total_floors = int(match.group(2))
        # 判断低楼层、中楼层和高楼层
        if floor_num <= total_floors * 0.3:  # 低楼层
            return '低楼层'
        elif floor_num <= total_floors * 0.7:  # 中楼层
            return '中楼层'
        else:  # 高楼层
            return '高楼层'
    # 3. 处理 '低楼层/x楼'、'中楼层/x楼'、'高楼层/x楼' 格式的数据
    match = re.match(r'(低楼层|中楼层|高楼层)/(\d+)楼', floor_data)
    if match:
        return match.group(1)  # 返回对应的楼层类型
    
    # 对于其他情况，返回 '未知'
    return '未知'

# 应用转换函数
test_rent['楼层类型'] = test_rent['楼层'].apply(convert_floor)

# 将楼层类型转化为编码
floor_mapping = {'低楼层': 1, '中楼层': 2, '高楼层': 3, '地下室': 4, '未知': 5}
test_rent['楼层类型编码'] = test_rent['楼层类型'].map(floor_mapping)

# 查看结果
test_rent['楼层类型编码'].head()


0    5
1    5
2    5
3    5
4    5
Name: 楼层类型编码, dtype: int64

In [209]:
# 确保 '房屋户型' 列的数据为字符串格式，并填充缺失值
test_rent['户型'] = test_rent['户型'].fillna('未知').astype(str)

# 应用到数据
test_rent[['房间数', '室数', '厅数', '厨数', '卫数']] = test_rent['户型'].apply(lambda x: pd.Series(parse_layout_v2(x)))

# 检查处理后的数据
print(test_rent[['户型', '房间数', '室数', '厅数', '厨数', '卫数']].head())


       户型  房间数  室数  厅数  厨数  卫数
0  2室2厅1卫    0   2   2   0   1
1  2室1厅1卫    0   2   1   0   1
2  2室2厅1卫    0   2   2   0   1
3  2室1厅1卫    0   2   1   0   1
4  3室2厅2卫    0   3   2   0   2


In [210]:
# 获取 '有电梯' 列的众数
mode_value = test_price['有电梯'].mode()[0]

# 使用众数填充缺失值
test_price['有电梯'] = test_price['有电梯'].fillna(mode_value)

# 查看填充后的结果
print(test_price['有电梯'].head())


0    1.0
1    1.0
2    1.0
3    0.0
4    0.0
Name: 有电梯, dtype: float64


In [211]:
test_rent.drop(columns=['装修','租赁方式','电梯','燃气','环线位置'], inplace=True)
test_rent['城市'] = pd.to_numeric(test_rent['城市'], errors='coerce')
test_rent['面积（㎡）'] = pd.to_numeric(test_rent['面积（㎡）'], errors='coerce')
test_rent[['付款方式_双月付价','付款方式_季付价','付款方式_年付价','付款方式_月付价','付款方式_未知']] = test_rent[['付款方式_双月付价','付款方式_季付价','付款方式_年付价','付款方式_月付价','付款方式_未知']].astype(int)

# feature engineering

## skewness

In [212]:
test_price.columns

Index(['ID', '城市', '区域', '板块', '房屋户型', '建筑面积（㎡）', '套内面积', '房屋朝向', '建筑结构',
       '装修情况', '梯户比例', '别墅类型', '交易时间', '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属',
       '抵押信息', '房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', 'lon', 'lat', '年份',
       '区县', '板块_comm', '环线位置', '物业类别', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司',
       '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '建筑结构_comm', '物业办公电话', '产权描述', '供水',
       '供暖', '供电', '燃气费（元/m³）', '供热费（元/㎡）', '停车位（个）', '停车费用（元）', 'coord_x',
       'coord_y', '客户反馈', '环线编码', '楼层编码', '有电梯', '房间数', '室数', '厅数', '厨数', '卫数',
       '建筑结构编码'],
      dtype='object')

In [213]:
test_price['log_area'] = np.log1p(test_price['建筑面积（㎡）'])
test_price['log_ringcode'] = np.log1p(test_price['环线编码'])


In [214]:
test_rent['log_area'] = np.log1p(test_rent['面积（㎡）'])
test_rent['log_heating_fee'] = np.log1p(test_rent['供热费（元/㎡）'])

for col in ['log_area', 'log_heating_fee']:
    print(col, '偏度 =', test_rent[col].skew())

log_area 偏度 = -1.019713000417501
log_heating_fee 偏度 = -3.0081686334182387


## interaction

In [215]:
test_price['建筑面积_绿化率'] = test_price['建筑面积（㎡）'] * test_price['绿化率（%）']
test_price['城市_建筑面积'] = test_price['城市'] * test_price['建筑面积（㎡）']
test_price['板块_容积率'] = test_price['板块'] * test_price['容积率（倍）']
test_price['log_area_房间数'] = test_price['log_area'] * test_price['房间数']


In [216]:
test_rent['城市_面积'] = test_rent['城市'] * test_rent['面积（㎡）']
test_rent['整租_房间数'] = test_rent['整租'] * test_rent['房间数']
test_rent['有电梯_楼层类型'] = test_rent['有电梯'] * test_rent['楼层类型编码']


## binning

In [217]:
df1 = test_price.copy()

# 自定义分箱区间
bins = [0, 300, 400, df1['停车费用（元）'].max()]  # 分成三个区间：低、中、高
labels = ['低停车费', '中停车费', '高停车费']  # 对应标签
# 创建分箱列
df1['停车费用_bin'] = pd.cut(df1['停车费用（元）'], bins=bins, labels=labels, include_lowest=True)
# 查看结果
print(df1[['停车费用（元）', '停车费用_bin']].sample(5))
# One-Hot 编码
df1_dummies = pd.get_dummies(df1['停车费用_bin'], prefix='停车费用', drop_first=True)
df1 = pd.concat([df1, df1_dummies], axis=1)
# 查看编码后的结果
print(df1.sample(5))

       停车费用（元） 停车费用_bin
5067     250.0     低停车费
12046    320.0     中停车费
15303    250.0     低停车费
19489    250.0     低停车费
17079    250.0     低停车费
            ID  城市     区域      板块      房屋户型  建筑面积（㎡）    套内面积 房屋朝向  建筑结构 装修情况  \
3891   1003891   0    7.0   164.0  1室1厅1厨1卫    72.40     NaN    东  钢混结构   简装   
16885  1016885   3   33.0   867.0  3室2厅1厨2卫   112.47    101㎡    南  钢混结构   精装   
816    1000816   0  112.0  1030.0  2室1厅1厨1卫    61.04     NaN  南 北  混合结构   精装   
16839  1016839   3  126.0  1081.0  3室2厅1厨1卫    89.16     NaN    南  钢混结构   精装   
2552   1002552   0   68.0   876.0  2室1厅1厨1卫    80.54  65.21㎡    东  钢混结构   其他   

       ... 建筑结构编码  log_area log_ringcode 建筑面积_绿化率 城市_建筑面积   板块_容积率  \
3891   ...      1  4.295924     1.386294  2244.40    0.00   455.92   
16885  ...      1  4.731538     0.000000  3936.45  337.41  1994.10   
816    ...      3  4.127779     1.791759  1831.20    0.00  2266.00   
16839  ...      1  4.501586     1.098612  3120.60  267.48  2486.30   
2552   ...      1  4.4010

In [218]:
df1.drop(columns=['停车费用_bin','供热费（元/㎡）'],inplace=True)
df1['停车费用_中停车费'] = df1['停车费用_中停车费'].astype(int)  # 转换为整数 0 或 1
df1['停车费用_高停车费'] = df1['停车费用_高停车费'].astype(int) 

In [219]:
df2 = test_rent.copy()
# 自定义分箱
bins = [0, 20, 30, df2['供热费（元/㎡）'].max()]
labels = ['低供热费', '中供热费', '高供热费']

df2['供热费_bin'] = pd.cut(df2['供热费（元/㎡）'], bins=bins, labels=labels, include_lowest=True)

# One-Hot 编码
df2_dummies = pd.get_dummies(df2['供热费_bin'], prefix='供热费', drop_first=True)
df2 = pd.concat([df2, df2_dummies], axis=1)

# 查看结果
print(df2[['供热费（元/㎡）', '供热费_bin', '供热费_中供热费', '供热费_高供热费']].head())


   供热费（元/㎡） 供热费_bin  供热费_中供热费  供热费_高供热费
0      25.0    中供热费      True     False
1      25.0    中供热费      True     False
2      25.0    中供热费      True     False
3      30.0    中供热费      True     False
4      25.0    中供热费      True     False


In [220]:
df2.drop(columns=['面积（㎡）','供热费（元/㎡）','log_heating_fee','供热费_bin'],inplace=True)
df2['供热费_中供热费'] = df2['供热费_中供热费'].astype(int)  # 转换为整数 0 或 1
df2['供热费_高供热费'] = df2['供热费_高供热费'].astype(int) 

In [221]:
df1.columns

Index(['ID', '城市', '区域', '板块', '房屋户型', '建筑面积（㎡）', '套内面积', '房屋朝向', '建筑结构',
       '装修情况', '梯户比例', '别墅类型', '交易时间', '交易权属', '上次交易', '房屋用途', '房屋年限', '产权所属',
       '抵押信息', '房屋优势', '核心卖点', '户型介绍', '周边配套', '交通出行', 'lon', 'lat', '年份',
       '区县', '板块_comm', '环线位置', '物业类别', '建筑年代', '开发商', '房屋总数', '楼栋总数', '物业公司',
       '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）', '建筑结构_comm', '物业办公电话', '产权描述', '供水',
       '供暖', '供电', '燃气费（元/m³）', '停车位（个）', '停车费用（元）', 'coord_x', 'coord_y',
       '客户反馈', '环线编码', '楼层编码', '有电梯', '房间数', '室数', '厅数', '厨数', '卫数', '建筑结构编码',
       'log_area', 'log_ringcode', '建筑面积_绿化率', '城市_建筑面积', '板块_容积率',
       'log_area_房间数', '停车费用_中停车费', '停车费用_高停车费'],
      dtype='object')

In [222]:
df2.columns

Index(['ID', '城市', '户型', '楼层', 'lon', 'lat', '区县', '板块', '绿化率（%）', '容积率（倍）',
       '物业费（元/月/㎡）', '燃气费（元/m³）', '停车位（个）', '精装修', '付款方式_双月付价', '付款方式_季付价',
       '付款方式_年付价', '付款方式_月付价', '付款方式_未知', '整租', '有电梯', '有燃气', '环线编码', '楼层类型',
       '楼层类型编码', '房间数', '室数', '厅数', '厨数', '卫数', 'log_area', '城市_面积', '整租_房间数',
       '有电梯_楼层类型', '供热费_中供热费', '供热费_高供热费'],
      dtype='object')

# featuring selection

# modeling

In [223]:
test_price['log_stru_code'] = np.log1p(test_price['建筑结构编码'])

In [224]:
X_price = test_price[['城市', '区域', '板块', '建筑面积（㎡）', 'lon', '绿化率（%）', '容积率（倍）', '物业费（元/月/㎡）',
       '燃气费（元/m³）', '停车位（个）', '停车费用（元）', '环线编码', '楼层编码', '房间数', '厅数', '厨数',
       '卫数', '有电梯', 'log_area', 'log_ringcode', 'log_stru_code', '建筑面积_绿化率']]
X_rent = test_rent[['城市', '面积（㎡）', 'lon','容积率（倍）', '物业费（元/月/㎡）', '燃气费（元/m³）',
       '供热费（元/㎡）', '停车位（个）', '精装修', '付款方式_季付价', '付款方式_年付价', '付款方式_月付价',
       '付款方式_未知', '整租', '有燃气', '环线编码', '厅数', '楼层类型编码', '有电梯_楼层类型']]

In [231]:
import joblib
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. 加载训练好的模型和预处理器 
ohe_price = joblib.load("/kaggle/input/onehot/pytorch/default/1/onehot_encoder_price.pkl")
ohe_rent = joblib.load("/kaggle/input/onehot/pytorch/default/1/onehot_encoder_rent (1).pkl")
scaler_price = joblib.load("/kaggle/input/scaler2/pytorch/default/1/scaler_price.pkl")
scaler_rent = joblib.load("/kaggle/input/scaler2/pytorch/default/1/scaler_rent (1).pkl")

# 2. 测试集列定义
cat_cols_price = ['城市']  
num_cols_price = [c for c in X_price.columns if c not in cat_cols_price]  
cat_cols_rent = ['城市']  
num_cols_rent = [c for c in X_rent.columns if c not in cat_cols_rent]  

# 3. 如果测试集缺少训练集的数值列，补齐
for col in scaler_price.feature_names_in_:  # scaler保存的训练列
    if col not in X_price.columns:
        X_price[col] = 0
for col in scaler_rent.feature_names_in_:
    if col not in X_rent.columns:
        X_rent[col] = 0

# 4. 测试集分类特征编码（只 transform，不 fit）
X_price_cat = ohe_price.transform(X_price[cat_cols_price])
X_rent_cat = ohe_rent.transform(X_rent[cat_cols_rent])

# 5. 测试集数值特征标准化（只 transform，不 fit）
X_price_num = scaler_price.transform(X_price[num_cols_price].astype(float))
X_rent_num = scaler_rent.transform(X_rent[num_cols_rent].astype(float))

# 6. 合并特征
X_price_final = np.hstack([X_price_num, X_price_cat])
X_rent_final = np.hstack([X_rent_num, X_rent_cat])


In [232]:
# 加载训练好的模型
best_model1_OLS = joblib.load('/kaggle/input/ols5/pytorch/default/1/model_OLS_price.pkl')
best_model2_OLS = joblib.load('/kaggle/input/ols5/pytorch/default/1/model_OLS_rent (1).pkl')
# 使用模型进行预测
y_test_price_pred = best_model1_OLS.predict(X_price_final)
y_test_rent_pred = best_model2_OLS.predict(X_rent_final)

y_test_price_pred_original = np.exp(y_test_price_pred)
y_test_rent_pred_original = np.exp(y_test_rent_pred)

# 将预测的价格和租金分别与ID合并
df_price_pred = pd.DataFrame({
    'ID': df1['ID'],  # 房子的ID
    'Predicted_Price': y_test_price_pred_original  # 预测的价格
})

df_rent_pred = pd.DataFrame({
    'ID': df2['ID'],  # 房子的ID
    'Predicted_Rent': y_test_rent_pred_original  # 预测的租金
})

df_final_pred = pd.merge(df_price_pred, df_rent_pred, on='ID', how='outer')
df_final_pred.to_csv('/kaggle/working/results_pred_OLS.csv', index=False)


In [233]:
# 加载训练好的模型
best_model1_lasso = joblib.load('/kaggle/input/lasso5/pytorch/default/1/model_Lasso_price.pkl')
best_model2_lasso = joblib.load('/kaggle/input/lasso5/pytorch/default/1/model_Lasso_rent (1).pkl')
# 使用模型进行预测
y_test_price_pred = best_model1_lasso.predict(X_price_final)
y_test_rent_pred = best_model2_lasso.predict(X_rent_final)

y_test_price_pred_original = np.exp(y_test_price_pred)
y_test_rent_pred_original = np.exp(y_test_rent_pred)

# 将预测的价格和租金分别与ID合并
df_price_pred = pd.DataFrame({
    'ID': df1['ID'],  # 房子的ID
    'Predicted_Price': y_test_price_pred_original  # 预测的价格
})

df_rent_pred = pd.DataFrame({
    'ID': df2['ID'],  # 房子的ID
    'Predicted_Rent': y_test_rent_pred_original  # 预测的租金
})

df_final_pred = pd.merge(df_price_pred, df_rent_pred, on='ID', how='outer')
df_final_pred.to_csv('/kaggle/working/results_pred_lasso.csv', index=False)


In [234]:
# 加载训练好的模型
best_model1_ridge = joblib.load('/kaggle/input/ridge5/pytorch/default/1/model_Ridge_price.pkl')
best_model2_ridge = joblib.load('/kaggle/input/ridge5/pytorch/default/1/model_Ridge_rent (1).pkl')
# 使用模型进行预测
y_test_price_pred = best_model1_ridge.predict(X_price_final)
y_test_rent_pred = best_model2_ridge.predict(X_rent_final)

y_test_price_pred_original = np.exp(y_test_price_pred)
y_test_rent_pred_original = np.exp(y_test_rent_pred)

# 将预测的价格和租金分别与ID合并
df_price_pred = pd.DataFrame({
    'ID': df1['ID'],  # 房子的ID
    'Predicted_Price': y_test_price_pred_original  # 预测的价格
})

df_rent_pred = pd.DataFrame({
    'ID': df2['ID'],  # 房子的ID
    'Predicted_Rent': y_test_rent_pred_original  # 预测的租金
})

df_final_pred = pd.merge(df_price_pred, df_rent_pred, on='ID', how='outer')
df_final_pred.to_csv('/kaggle/working/results_pred_ridge.csv', index=False)


In [235]:
# 3. 加载训练好的模型
best_model1_ElasticNet = joblib.load('/kaggle/input/en5/pytorch/default/1/model_ElasticNet_price (1).pkl')  # 价格模型
best_model2_ElasticNet = joblib.load('/kaggle/input/en5/pytorch/default/1/model_ElasticNet_rent.pkl')  # 租金模型

# 4. 使用价格模型进行预测
y_test_price_pred = best_model1_ElasticNet.predict(X_price_final)

# 5. 使用租金模型进行预测
y_test_rent_pred = best_model2_ElasticNet.predict(X_rent_final)

# 6. 对预测结果进行反对数操作（因为模型输出的是对数价格/租金）
y_test_price_pred_original = np.exp(y_test_price_pred)
y_test_rent_pred_original = np.exp(y_test_rent_pred)

# 7. 合并预测结果与 ID
df_price_pred = pd.DataFrame({
    'ID': df1['ID'],  # 假设 df1 存储了与价格相关的 ID
    'Predicted_Price': y_test_price_pred_original  # 预测的价格
})

df_rent_pred = pd.DataFrame({
    'ID': df2['ID'],  # 假设 df2 存储了与租金相关的 ID
    'Predicted_Rent': y_test_rent_pred_original  # 预测的租金
})

# 8. 合并价格和租金预测结果
df_final_pred = pd.merge(df_price_pred, df_rent_pred, on='ID', how='outer')

# 9. 保存预测结果到CSV文件
df_final_pred.to_csv('/kaggle/working/results_pred_ElasticNet2.csv', index=False)
